# Simple RAG with LangChain (Python 3.11.9)

This notebook loads five realistic, fictional training PDFs from the `sample_pdfs` folder and uses them as a RAG knowledge base. Each PDF has three pages of document text.

**Flow:** load PDFs → split text → index chunks → retrieve relevant text → answer questions.

Keep this notebook beside the `sample_pdfs` folder and run cells from top to bottom. You need an OpenAI API key.

## 1. Install packages and set your API key

Run the install cell once. Restart the kernel if prompted. Add your key to a `.env` file beside this notebook:

```text
OPENAI_API_KEY=your-key-here
```

Keep your real key private.

In [ ]:
%pip install -q langchain langchain-classic langchain-community langchain-openai langchain-chroma langchain-text-splitters chromadb pypdf python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Add OPENAI_API_KEY to a .env file or your environment, then rerun this cell.")

CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

## 2. Put the PDF files beside the notebook

Place the provided `sample_pdfs` folder in the same directory as this notebook. It should contain `company_overview.pdf`, `leave_policy.pdf`, `expense_policy.pdf`, `security_policy.pdf`, and `remote_work_policy.pdf`. This notebook reads those files directly; it does not generate them.

## 3. Load all five PDF files

Each page becomes a document, with the PDF filename and page number kept as metadata.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_dir = Path("sample_pdfs")
pdf_files = sorted(pdf_dir.glob("*.pdf"))
if len(pdf_files) != 5:
    raise FileNotFoundError("Expected five PDF files in the sample_pdfs folder. Put that folder beside the notebook.")

pages = []
for pdf_file in pdf_files:
    pages.extend(PyPDFLoader(str(pdf_file)).load())

print("PDF files:", len(pdf_files))
print("Pages loaded:", len(pages))
for pdf_file in pdf_files:
    page_count = sum(1 for page in pages if Path(page.metadata["source"]).name == pdf_file.name)
    print(f"{pdf_file.name}: {page_count} pages")
    assert page_count >= 2, f"{pdf_file.name} should have at least two pages"


## 4. Split documents into chunks

Chunking breaks each PDF into smaller pieces that can be searched.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print("Chunks created:", len(chunks))
print("Example chunk:", chunks[0].page_content[:300])

## 5. Create embeddings and store chunks in Chroma

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="company_documents",
    persist_directory="chroma_db"
)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
print("Vector database is ready.")

## 6. Build the RAG question-answer chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the context below. If the answer is missing, say you could not find it.\n\nContext:\n{context}"),
    ("human", "{input}")
])
answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, answer_chain)

## 7. Ask questions about the PDFs

In [ ]:
def ask(question):
    result = rag_chain.invoke({"input": question})
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Source files:", sorted({Path(doc.metadata["source"]).name for doc in result["context"]}))
    print()
    return result

ask("How many paid annual leave days do full-time employees get?")

In [ ]:
ask("When should employees submit expense claims?")
ask("What should I do if I receive a phishing message?")
ask("How many days per week can an employee work remotely?")

## Summary

The separate PDF files are the knowledge sources. The loader reads them, the splitter creates searchable chunks, Chroma retrieves relevant text, and the chat model answers using those chunks. The PDFs are provided as files, so no document-content dictionary is needed in this notebook.